# Report — assignment 2: Learn to reconstruct images from CAMUS dataset

**IAV — INSA Lyon**

Tayeb BOUCHIKHI

Nine experiments, one question: can a VAE rebuild a 64×64 ultrasound frame, and can one axis of its latent code follow the size of the left-ventricle cavity?

We keep **experiment 9**. On the 200 test frames: MSE 0.00221, SSIM 0.831, PSNR 26.76 dB, Spearman 0.953 between the first latent axis and the cavity area.

Every figure below is copied from the experiment notebook that produced it. Nothing was replotted. A ± value is the standard deviation across the 200 test frames, not a confidence interval.


## Contents

1. [Use of AI](#ai)
2. [Question](#question)
3. [Data](#data)
4. [Network](#network)
5. [Scores](#scores)
6. [What stayed fixed](#protocol)
7. [The nine experiments](#experiments)
8. [Comparison](#comparison)
9. [What we keep](#decision)
10. [Bibliography](#bibliography)


<a id="ai"></a>

## 1. Use of AI

**What the assistant produced**

- The training loops : the weighted loss, the gradient term, the three-attribute loss, and the exponential moving average of the weights.
- Translation from French to English.

**What we decided, and what we check**

- The patient split, the seed, and the choice to decode μ at test time.
- The order of the experiments: first reconstruct, then make one axis follow the cavity, then spend capacity on sharpness.
- Every number in the tables. They are copied from the run outputs. They are not estimated.


<a id="question"></a>

## 2. Question

A frame is a grey sector, a dark cavity, a bright wall, and a lot of speckle. Between end-diastole (ED) and end-systole (ES), what changes most is the size of the cavity. We want both jobs at once.

1. **Rebuild the frame**, wall and speckle included.
2. **Put the cavity size on one axis.** If the first latent channel ranks the frames in the same order as the left-ventricle area, that axis can be read as a cardiac attribute. The other channels stay free (view, texture, the rest).

The test set is 50 patients held out from the start. It is read once the checkpoint has already been chosen on validation.


<a id="data"></a>

## 3. Data

CAMUS gives, for each patient, a 2-chamber view and a 4-chamber view, each at ED and at ES. Four frames per patient, already 64×64. The expert mask:

| Label | Structure |
| --- | --- |
| 0 | background |
| 1 | left-ventricle cavity |
| 2 | myocardium |
| 3 | left atrium |

Intensities above 1 are divided by 255. We do not divide by the maximum of each image. That would brighten a dark exam and make the losses incomparable across patients.

The split is by **patient**, seed 12345, and it is the same split in every experiment.

| Set | Patients | Frames | Role |
| --- | ---: | ---: | --- |
| Train | 400 | 1,600 | weight updates |
| Validation | 50 | 200 | choose the epoch |
| Test | 50 | 200 | one read, checkpoint already frozen |


<img src="report_figures/exp9_batch.png" alt="Ten training frames" width="900" style="max-width:100%; height:auto;">

*Ten training frames, apex up. The same intensity and view check is in every notebook.*

<img src="report_figures/exp9_weights.png" alt="Image, mask and weights" width="820" style="max-width:100%; height:auto;">

*Image, expert mask, weight map W. The mask is not an input of the network. It only weights the loss: here a heart pixel counts as 4.5, a background pixel as 1.*

<a id="network"></a>

## 4. Network

The encoder takes the image from 64×64 down to 8×8 and predicts two maps, μ and σ. At training time we sample

$$
z = \mu + \varepsilon \sigma, \qquad \varepsilon \sim \mathcal{N}(0, I).
$$

At test time we decode μ. A random draw would change the MSE on every measurement. `AutoencoderKL` samples even in `eval` mode, so the notebooks call `encode`, then `decode(μ)`.

A 4×8×8 code has 256 numbers for 4,096 pixels (compression ×16). An 8×8×8 code has 512 (×8). A 16×8×8 code has 1,024 (×4). A short code forces the network to throw information away. That helps a clean latent space, and it hurts if we want the speckle back.

Without a penalty, σ can collapse and two similar hearts end up far apart. An ED → ES interpolation then walks through codes the decoder has never seen. The KL pulls every posterior toward the standard normal:

$$
\mathrm{KL} = \frac{1}{2}\sum_k \left(\mu_k^2 + \sigma_k^2 - \log \sigma_k^2 - 1\right).
$$

The loss is reconstruction plus β · KL. If β is too large, the image becomes a blur. Experiment 1 shows it.

From experiment 3 on, a ranking term asks \(z_0\) to order the frames the same way as the cavity areas. For every pair in the batch:

$$
L_{\mathrm{attr}} = \mathrm{mean}_{i \neq j} \big| \tanh\big(\delta (z_{0,i} - z_{0,j})\big) - \mathrm{sign}(a_i - a_j) \big|,
$$

with δ = 2. The loss does not care by how much \(z_0\) moves, only by the direction. γ multiplies this term. Spearman is the matching score: both the loss and Spearman care about rank, not about a slope.


<a id="scores"></a>

## 5. Scores

Everything is computed by decoding μ, with frames in [0, 1].

| Score | What it means on these images |
| --- | --- |
| MSE | Average squared error. Lower is better. |
| MAE | Average absolute error, easier to picture as a grey-level gap. |
| PSNR | The same ranking as MSE, in dB. |
| SSIM | Structure in a 7×7 window. |
| Heart MSE | MSE on `label > 0`. This is the number that matters: the background is easy. |
| Spearman(z0, LV area) | Rank agreement between the first axis and the cavity fraction. |

The checkpoint is not chosen on the test set. It is the epoch, after the β warm-up, with the lowest validation reconstruction.

MSE and SSIM can disagree. Experiment 2 has a better MSE than experiment 1 (0.00765 against 0.01024) and a worse SSIM (0.445 against 0.622). From experiment 4 on, the loss includes SSIM so that this trade-off is not left to chance.


<a id="protocol"></a>

## 6. What stayed fixed

- Seed 12345, patient-wise 400 / 50 / 50 split, intensities in [0, 1].
- 2D `AutoencoderKL`, one input channel, GroupNorm when the widths allow it.
- Adam or AdamW, cosine learning-rate decay.
- Test reconstruction = decoding of μ.
- The four frames: `2CH_ED`, `2CH_ES`, `4CH_ED`, `4CH_ES`.

What we allowed ourselves to change, one experiment at a time: the width, the length of the code, the loss, β, γ, the clip, an exponential moving average of the weights, the number of epochs, and the device (CPU for 1–3, an RTX 4060 Ti for 4–9).


<a id="experiments"></a>

## 7. The nine experiments

### 7.1 Experiment 1 — a first VAE, on CPU

Notebook `experiments/cpu/exp_1_vae_camus.ipynb`. Channels `(16, 32, 64, 64)`, one residual block per scale, latent **4×8×8**, 473,649 parameters. The reconstruction term is an L1. β is chosen on validation only, over {0, 10⁻⁴, 10⁻³, 10⁻², 10⁻¹}, for 8 epochs each.

The rule keeps the largest β that stays within 0.4 dB of the best PSNR among β > 0. That is β = 10⁻⁴. At 10⁻³ the KL has already won and the heart is gone.


<img src="report_figures/exp1_beta_metrics.png" alt="Beta ablation" width="980" style="max-width:100%; height:auto;">

*Validation, 8 epochs. In blue, β = 10⁻⁴. PSNR stays close to β = 0, and the KL falls from about 1,000 to about 70.*

<img src="report_figures/exp1_beta_recon.png" alt="Reconstructions against beta" width="520" style="max-width:100%; height:auto;">

*The same four validation frames. β = 0 and β = 10⁻⁴ stay readable. From 10⁻³ on, every frame looks the same.*

The selected VAE is then trained for 25 epochs. Test, 200 frames: **MSE 0.01024, MAE 0.057, PSNR 20.97 dB, SSIM 0.622, heart MSE 0.01511**. A mean frame sits at 17.75 dB, so the network does more than predict the average sector. A deterministic autoencoder with the same backbone reaches 21.77 dB, and a VQ-VAE reaches 20.82 dB. The deterministic model is slightly sharper because it never pays for the KL. There is no attribute loss yet, so no Spearman is reported.

26.8 minutes on CPU. The best epoch is the last one (25): the model had not finished improving.


<img src="report_figures/exp1_curves.png" alt="VAE training curves, experiment 1" width="980" style="max-width:100%; height:auto;">

*Training of the selected VAE, β = 10⁻⁴. The reconstruction L1 is still falling at epoch 25.*

<img src="report_figures/exp1_compare.png" alt="AE, VAE and VQ-VAE" width="560" style="max-width:100%; height:auto;">

*Same test frames. The deterministic autoencoder keeps a bit more of the wall. The VQ-VAE of this run uses 2 codes out of 128: the codebook collapsed.*

<img src="report_figures/exp1_best.png" alt="Best reconstructions, experiment 1" width="680" style="max-width:100%; height:auto;">

*The three best test reconstructions of this VAE. The error is still large on the wall, and the speckle is not back.*

<img src="report_figures/exp1_worst.png" alt="Worst reconstructions, experiment 1" width="680" style="max-width:100%; height:auto;">

*The three worst. The MSE standard deviation (0.01093) is as large as the mean: some frames are far off.*

### 7.2 Experiment 2 — a wider code, still only MSE + KL

Notebook `experiments/cpu/exp_2_vae_camus.ipynb`. Channels `(32, 64, 128, 128)`, latent **8×8×8**, 1,888,449 parameters. Loss MSE + β KL, β fixed at 10⁻⁴, 20 epochs.

Test: **MSE 0.00765, MAE 0.055, PSNR 21.70 dB, SSIM 0.445, heart MSE 0.01073**. Best epoch 19. Duration 30.5 minutes.

Pixels get closer, local structure gets worse. A wider network trained on MSE alone can wash out the edges the SSIM window is looking at. This is why the later losses are not pure MSE.


<img src="report_figures/exp2_curves.png" alt="Experiment 2 curves" width="980" style="max-width:100%; height:auto;">

*Training MSE on a noisy z, validation MSE on μ. Both fall. Test SSIM does not follow.*

<img src="report_figures/exp2_recon.png" alt="Experiment 2 reconstructions" width="980" style="max-width:100%; height:auto;">

*Ten test frames and their reconstruction. The sector is there. The speckle and the thin walls are not.*

### 7.3 Experiment 3 — the axis appears, too weakly

Notebook `experiments/cpu/exp_3_vae_camus.ipynb`. Back to a small network, `(16, 32, 64, 64)`, latent 8×8×8, 478,417 parameters, so the CPU run stays short (12.6 minutes, 20 epochs). The new term is the ranking loss of section 4, with γ = 10⁻³ and δ = 2. β warms up linearly to 10⁻⁴ over 5 epochs.

Test: **MSE 0.01024, MAE 0.061, PSNR 20.84 dB, SSIM 0.703, heart MSE 0.01298, Spearman(z0, LV area) = 0.023**.

SSIM comes back, because the network is smaller and less free to blur in a way that only MSE rewards. The Spearman is what you get from an untrained axis. γ = 10⁻³ does not compete with the reconstruction term. The idea is right. The weight is not.


<img src="report_figures/exp3_curves.png" alt="Experiment 3 curves" width="980" style="max-width:100%; height:auto;">

*Validation reconstruction sits below training reconstruction, which is expected: validation decodes μ, training decodes z.*

<img src="report_figures/exp3_recon.png" alt="Experiment 3 reconstructions" width="900" style="max-width:100%; height:auto;">

*Ten test frames. Less washed out than experiment 2, still far from a sharp ultrasound frame.*

<img src="report_figures/exp3_latent.png" alt="Experiment 3 latent space" width="980" style="max-width:100%; height:auto;">

*On the right, z0 against cavity area. The cloud is flat: Spearman 0.02. The axis is named, it is not trained.*

### 7.4 Experiment 4 — GPU, a weighted loss, a real axis

Notebook `experiments/gpu/exp_4_vae_camus.ipynb`. RTX 4060 Ti.

Channels `(32, 64, 128, 128)`, two residual blocks, latent 8×8×8, no attention, 3,256,641 parameters. The loss becomes

$$
L = 0.50\, L_{\mathrm{mse},w} + 0.25\, L_{1,w} + 0.25\,(1 - \mathrm{SSIM}) + \beta\, \mathrm{KL} + \gamma\, L_{\mathrm{attr}}.
$$

\(W = 1 + 3\cdot\mathbf{1}(\mathrm{label}>0)\): a heart pixel counts as 4. β warms up to 5×10⁻⁵ over 10 epochs. γ = 0.05. AdamW at 10⁻³, cosine down to 10⁻⁵, 80 epochs, FP32.

Test: **MSE 0.00343 ± 0.00261, MAE 0.034 ± 0.010, PSNR 25.12 ± 1.70 dB, SSIM 0.787 ± 0.030, heart MSE 0.00518 ± 0.00206, Spearman 0.943**. Epoch 76. 3.75 minutes, about 2.5 GB of VRAM.

This is the first run that looks like ultrasound, and the first whose axis follows the cavity (p about 10⁻⁹⁶). The checks written in the notebook all pass (MSE < 0.007, PSNR > 22 dB, SSIM > 0.72, heart MSE < 0.0095, Spearman > 0.85). The optional time budget of 120 seconds does not: the run took 225 seconds. We did not shrink the model to win that clock.


<img src="report_figures/exp4_curves.png" alt="Experiment 4 curves" width="980" style="max-width:100%; height:auto;">

*MSE, SSIM and the attribute loss. The attribute falls, SSIM rises, and the checkpoint is near the end (epoch 76).*

<img src="report_figures/exp4_recon.png" alt="Experiment 4 reconstructions" width="900" style="max-width:100%; height:auto;">

*Ten test frames. The cavity and the wall are in the right place. This is the jump from the CPU runs.*

<img src="report_figures/exp4_latent.png" alt="Experiment 4 latent space" width="980" style="max-width:100%; height:auto;">

*z0 against cavity area, Spearman 0.94. t-SNE partly groups the views, but the four colours follow the same line: the axis follows the area, not only the view.*

### 7.5 Experiment 5 — the reference

Notebook `experiments/gpu/exp_5_vae_camus.ipynb`.

Same 8×8×8 grid. The last stage widens from 128 to 256 channels, and an attention block is added on the 8×8 grid. 8,233,153 parameters. A weighted gradient term is added, for the endocardial and epicardial edges:

$$
L_{\mathrm{recon}} = 0.40\, L_{\mathrm{mse},w} + 0.25\, L_{1,w} + 0.25\,(1-\mathrm{SSIM}) + 0.10\, L_{\nabla,w}.
$$

\(W = 1 + 3.5\cdot\mathbf{1}(\mathrm{label}>0)\). β warms up to 2×10⁻⁵ over 15 epochs. γ = 0.03, on \(z_0\) and the cavity only. AdamW at 10⁻³, cosine down to 10⁻⁵, 100 epochs, batch 64, FP32.

The checkpoint is epoch 48, the lowest validation reconstruction after the warm-up. The curve does not beat it through epoch 100. Duration 6.3 minutes, peak VRAM about 3.0 GB.

| | Mean | Std |
| --- | ---: | ---: |
| MSE | 0.00268 | 0.00138 |
| Heart MSE | 0.00446 | 0.00159 |
| Background MSE | 0.00212 | 0.00149 |
| MAE | 0.0277 | 0.0061 |
| PSNR | 26.04 dB | 1.50 dB |
| Heart PSNR | 23.73 dB | 1.33 dB |
| SSIM | 0.808 | 0.022 |

The heart is harder than the background. Heart PSNR is about 2.3 dB below the full-image PSNR.

| Attribute | In the loss? | Spearman with \(z_0\) |
| --- | --- | ---: |
| LV cavity (`label == 1`) | yes | **0.933** |
| Epicardial area (cavity + myocardium) | no | 0.882 |
| Left atrium (`label == 3`) | no | −0.221 |

The atrium goes the other way (p = 0.0016). During the cycle the atrium and the ventricle often move in opposite directions, and nothing in the loss asked \(z_0\) to represent the atrium. Inside each view the cavity Spearman stays high: the axis is not a disguised 2CH / 4CH label.

Experiment 5 beats experiment 4 on MSE, heart MSE, SSIM and PSNR. Spearman stays above 0.85 (0.933 against 0.943): a small giveaway, in exchange for a sharper image. This was the model we would have submitted, until experiment 9.


<img src="report_figures/exp5_curves.png" alt="Experiment 5 curves" width="980" style="max-width:100%; height:auto;">

*The checkpoint (dashed) is epoch 48. After that, validation MSE does not improve.*

<img src="report_figures/exp5_recon.png" alt="Experiment 5 reconstructions" width="900" style="max-width:100%; height:auto;">

*Ten test frames. Wall and speckle are much closer to the original than in experiment 4.*

<img src="report_figures/exp5_error.png" alt="Experiment 5 error maps" width="980" style="max-width:100%; height:auto;">

*patient0010, one row per view / instant. Columns: original, reconstruction with the expert contours, absolute error ×5, intensity profile along the cyan line. The error sits on the wall, not on the background.*

<img src="report_figures/exp5_latent.png" alt="Experiment 5 latent space" width="980" style="max-width:100%; height:auto;">

*z0 follows the cavity area, Spearman 0.93. The four view / instant pairs lie on the same line.*

<img src="report_figures/exp5_interp.png" alt="ED to ES interpolation, experiment 5" width="980" style="max-width:100%; height:auto;">

*patient0010, 4-chamber view, both frames are in the test set. The path links μ(ED) to μ(ES). The frame stays an ultrasound image all the way.*

<img src="report_figures/exp5_sweep.png" alt="z0 sweep, experiment 5" width="980" style="max-width:100%; height:auto;">

*Only the mean of channel 0 is moved, from a low value to a high one. The rest of the code is that of a test frame with a median cavity area.*

### 7.6 Experiment 6 — more capacity, and a clear miss

Notebook `experiments/gpu/exp_6_vae_camus.ipynb`.

The bet was that experiment 5 had plateaued at epoch 48 because the code was too short. This run removes the limits we had set.

- Channels `(64, 128, 256, 512)`, three residual blocks, attention at 16×16 and at 8×8. **49,122,305 parameters.**
- Latent **16×8×8**. Channels 0, 1 and 2 are supervised by the cavity, the myocardium and the atrium. Channels 3 to 15 are free.
- Loss: 0.35 weighted MSE, 0.25 weighted L1, 0.25 two-scale SSIM (64 and 32), 0.10 gradient, 0.05 L1 on the FFT amplitude. \(W = 1 + 4\cdot\mathbf{1}(\mathrm{label}>0)\).
- β warms up to 10⁻⁵ over 20 epochs. γ = 0.025 is multiplied by the mean of the three attribute terms, so about 0.008 per axis.
- 150 epochs, batch 32, AdamW at 8×10⁻⁴, clip at 1. From epoch 20, the exponential moving average of the weights (decay 0.995) is the one evaluated.

The first launch did not train: at initialization about half of the decoder pixels fell outside [0, 1], and the clamp killed their gradient. After the straight-through clamp the run takes 39.3 minutes, peak VRAM 5.8 GB. It is not a good model. The best checkpoint is epoch 150, and the validation MSE is still 0.00533. The attribute term stays near 0.96, its value at random.

| | Experiment 6 |
| --- | ---: |
| MSE | 0.00609 ± 0.00281 |
| Heart MSE | 0.00842 ± 0.00315 |
| MAE | 0.0422 ± 0.0094 |
| PSNR | 22.49 ± 1.63 dB |
| Heart PSNR | 21.00 ± 1.46 dB |
| SSIM | 0.682 ± 0.033 |
| Spearman(z0, cavity) | 0.156 |
| Spearman(z1, myocardium) | 0.018 |
| Spearman(z2, atrium) | 0.220 |

Why the extra capacity did not help. The clip at 1 is the same number as on the 8 M parameter model. Here the gradient norm starts near 12, so every step is cut by about that factor. The learning rate on the page is 8×10⁻⁴, and the step the weights take is much smaller. And 0.008 of γ per axis is too small: the ranking term never leaves 0.96.

The targets we had written down (MSE < 0.0018, heart MSE < 0.003, SSIM > 0.85, PSNR > 27.5 dB, Spearman > 0.85 on the three axes) all fail. We do not average this run with experiment 5, and we do not retune it on the test set.


<img src="report_figures/exp6_curves.png" alt="Experiment 6 curves" width="980" style="max-width:100%; height:auto;">

*150 epochs, and validation MSE is still high when the learning rate has already finished decaying. The attribute term (AR-VAE) never leaves 1.*

<img src="report_figures/exp6_error.png" alt="Experiment 6 error maps" width="980" style="max-width:100%; height:auto;">

*Same patient as experiment 5. The reconstruction is smooth, the error map is bright, and the orange profile misses the wall peaks on the 4-chamber views.*

<img src="report_figures/exp6_matrix.png" alt="Experiment 6 Spearman matrix" width="460" style="max-width:100%; height:auto;">

*Test-set Spearman, three axes against three areas. Nothing on the diagonal exceeds 0.22.*

### 7.7 Experiment 7 — the experiment 5 network, with the axes repaired

Notebook `experiments/gpu/exp_7_vae_camus.ipynb`.

Experiment 6 missed for three reasons that show up in the curve, not only in the table: 49 M parameters for 1,600 images, a clip of 1 on a norm that starts near 12, and a γ divided by three.

Experiment 7 does not add capacity. It copies the experiment 5 template and changes those three pieces. The straight-through clamp stays.

- Channels `(32, 64, 128, 256)`, two blocks, attention only at 8×8. **8,251,841 parameters.**
- Latent **12×8×8**. Channels 0, 1, 2: cavity, myocardium, atrium. Channels 3 to 11 stay free.
- Reconstruction, no FFT term: 0.40 MSE, 0.20 L1, 0.25 two-scale SSIM, 0.15 gradient. \(W = 1 + 4\cdot\mathbf{1}(\mathrm{label}>0)\).
- β up to \(1.5\times 10^{-5}\) over 15 epochs. Separate weights, not averaged: 0.030, 0.015, 0.015.
- 120 epochs, batch 64, AdamW at \(10^{-3}\), clip at 3. Exponential moving average (0.995) from epoch 15.

The best checkpoint is epoch 120, validation reconstruction 0.0746. Duration 7.8 minutes, about 3.0 GB. The curve is not smooth: around epoch 38 the EMA reconstruction jumps to 0.34, then climbs back. Final validation MSE is still 0.00497, above experiment 5.

| | Experiment 7 |
| --- | ---: |
| MSE | 0.00716 ± 0.01045 |
| Heart MSE | 0.00942 ± 0.01082 |
| MAE | 0.044 ± 0.022 |
| PSNR | 22.47 ± 2.18 dB |
| Heart PSNR | 21.04 ± 2.01 dB |
| SSIM | 0.715 ± 0.046 |
| Spearman(z0, cavity) | 0.918 |
| Spearman(z1, myocardium) | 0.849 |
| Spearman(z2, atrium) | 0.702 |

The image beats neither experiment 5 nor, on MSE, experiment 6. The image checks fail. Spearman(z0) > 0.85 passes. Inside each view, Spearman(z0, cavity) is 0.911 on 2CH and 0.924 on 4CH.

The three corrections did what they were aimed at for the latent code. They did not bring the image back. We do not retune this run on the test set.


<img src="report_figures/exp7_curves.png" alt="Experiment 7 curves" width="980" style="max-width:100%; height:auto;">

*The spike near epoch 38 is in the MSE and in the gradient. The model recovers, without getting back to experiment 5. The checkpoint is the last epoch.*

<img src="report_figures/exp7_error.png" alt="Experiment 7 error maps" width="980" style="max-width:100%; height:auto;">

*Same four frames. The 4-chamber views are blurry, and the profile misses both wall peaks. The MSE standard deviation (0.01045) is back to the size of the CPU runs.*

<img src="report_figures/exp7_matrix.png" alt="Experiment 7 Spearman matrix" width="460" style="max-width:100%; height:auto;">

*The diagonal is the strong entry of each row: 0.92, 0.85, 0.70. The axes are there. The image, on the figure above, is not.*

### 7.8 Experiment 8 — GPU augmentation, then a decoder trained alone on μ

Notebook `experiments/gpu/exp_8_vae_camus.ipynb`.

Experiment 5 reaches its best validation score at epoch 48. The decoder is trained on \(z = \mu + \varepsilon\sigma\) and tested on μ. Experiment 8 keeps the experiment 5 network (8,233,153 parameters, 8×8×8 code) and changes the training.

- Loss: 0.45 MSE, 0.20 L1, 0.25 SSIM, 0.10 gradient. \(W = 1 + 3.2\cdot\mathbf{1}(\mathrm{label}>0)\).
- β up to \(1.5\times 10^{-5}\). Weights 0.030 on the cavity, 0.005 on the myocardium, 0.005 on the atrium.
- Phase 1, epochs 1 to 140. For 110 epochs, each frame is warped on the GPU with probability 0.75: translation of at most 2 pixels, zoom in [0.96, 1.04], rotation in [−4°, +4°], gain in [0.96, 1.04]. The decoder sees \(z = \mu + \varepsilon\sigma\). Best phase-1 validation: epoch 138, 0.0750.
- Phase 2, epochs 141 to 200. The encoder is frozen. Only the decoder is trained, on μ, with no augmentation. Global best validation: epoch 193, 0.0721. Duration 11.6 minutes, about 3.0 GB.

| | Experiment 8 |
| --- | ---: |
| MSE | 0.00439 ± 0.00297 |
| Heart MSE | 0.00639 ± 0.00271 |
| MAE | 0.0356 ± 0.0098 |
| PSNR | 23.99 ± 1.63 dB |
| Heart PSNR | 22.21 ± 1.42 dB |
| SSIM | 0.765 ± 0.028 |
| Spearman(z0, cavity) | 0.930 |
| Spearman(z1, myocardium) | 0.880 |
| Spearman(z2, atrium) | 0.642 |

The image does not beat experiment 5: about 2 dB lower. Spearman(z0) passes, and so does the myocardium (0.880). The atrium stays at 0.642, under 0.70. Freezing the encoder kept the cavity axis. Polishing the decoder on μ moved validation from 0.0750 to 0.0721, still above experiment 5's 0.0613.


<img src="report_figures/exp8_curves.png" alt="Experiment 8 curves" width="980" style="max-width:100%; height:auto;">

*At epoch 110 the bilinear augmentation stops: the training gradient L1 jumps, because the real frames are sharper than the warped ones. Phase 2 (orange dotted line) does not close that gap.*

<img src="report_figures/exp8_error.png" alt="Experiment 8 error maps" width="980" style="max-width:100%; height:auto;">

*Same patient. Better than experiment 7, less sharp than experiment 5: the 4-chamber wall peaks are rounded.*

<img src="report_figures/exp8_matrix.png" alt="Experiment 8 matrices" width="860" style="max-width:100%; height:auto;">

*Spearman on the left, Pearson on the right. Cavity 0.93 and myocardium 0.88 line up. The atrium stops at 0.64.*

### 7.9 Experiment 9 — the experiment 5 loss, then a decoder trained on μ

Notebook `experiments/gpu/exp_9_vae_camus.ipynb`.

Experiment 8 lost about 2 dB. Three reasons, and experiment 9 is built to avoid each of them.

Through epoch 110, every augmented frame went through a bilinear `grid_sample`. On a 64×64 grid, a small rotation and a small zoom smooth the speckle and the myocardial edge. At epoch 111 the real frames come back, the learning rate is already down to \(1.18\times 10^{-4}\), and the encoder no longer has the step size to relearn the sharp textures. Freezing the encoder at epoch 141 then locked that code. The fine detail was already gone from μ.

The loss had also moved: less L1, less weight on the heart, and a ranking term on the atrium, a structure that is tiny or absent on some views. Spearman(z2) had stopped at 0.642.

Experiment 9 keeps the experiment 5 network and the experiment 5 loss. It changes the schedule.

- 8,233,153 parameters, latent 8×8×8, attention at 8×8.
- \(0.40\) MSE, \(0.25\) L1, \(0.25\) SSIM, \(0.10\) gradient. \(W = 1 + 3.5\cdot\mathbf{1}(\mathrm{label}>0)\).
- β up to \(2\times 10^{-5}\) over 15 epochs. γ = 0.03 on \(z_0\) and the cavity only.
- Phase 1, epochs 1 to 80. AdamW at \(10^{-3}\), the experiment 5 cosine schedule, clip at 1. Each training frame is shifted by an integer of at most 2 pixels, with zeros on the border that leaves the frame, and a gain in [0.96, 1.04], with probability 0.5. No rotation, no zoom, no `grid_sample`. The decoder sees \(z = \mu + \varepsilon\sigma\). Best validation: epoch 80, 0.0580, already under experiment 5's 0.0613.
- Phase 2, epochs 81 to 110. The encoder is not frozen. It restarts at \(10^{-4}\), the decoder at \(3\times 10^{-4}\). The same integer shift stays on. The training code is μ. Best validation: epoch 109, 0.0553. Duration 6.9 minutes, about 3.0 GB.

At epoch 80 the training reconstruction is 0.0597, SSIM 0.821, gradient 0.0462. At epoch 81, the first step on μ, those three numbers are 0.0710, 0.790 and 0.0491, and the decoder learning rate is back at \(3\times 10^{-4}\). The step is the change of code, from \(\mu + \varepsilon\sigma\) to μ. It is not the bilinear cutoff of experiment 8: there, the learning rate had no restart.

| | Experiment 9 |
| --- | ---: |
| MSE | 0.00221 ± 0.00075 |
| Heart MSE | 0.00375 ± 0.00123 |
| Background MSE | 0.00172 ± 0.00076 |
| MAE | 0.0252 ± 0.0040 |
| PSNR | 26.76 ± 1.29 dB |
| Heart PSNR | 24.46 ± 1.27 dB |
| SSIM | 0.831 ± 0.018 |
| Spearman(z0, cavity) | 0.953 |
| Spearman(z1, myocardium) | 0.669 |
| Spearman(z2, atrium) | −0.315 |

The image beats experiment 5 on the four checks: MSE, heart MSE, SSIM, PSNR. Spearman(z0) is 0.953, above 0.85, above experiment 5 (0.933) and above experiment 4 (0.943). Inside each view: 0.951 on 2CH and 0.951 on 4CH. Pearson on the same pair: 0.917.

\(z_1\) and \(z_2\) were measured, not trained. There is no atrium axis. Leaving \(z_2\) out of the loss is the third correction: experiment 8 spent gradient on that ranking and stopped at 0.642, while the frame got worse.


<img src="report_figures/exp9_curves.png" alt="Experiment 9 curves" width="980" style="max-width:100%; height:auto;">

*Phase 2 starts at epoch 80 (orange dotted line). Validation MSE keeps falling. The checkpoint is epoch 109.*

<img src="report_figures/exp9_recon.png" alt="Experiment 9 reconstructions" width="900" style="max-width:100%; height:auto;">

*Ten test frames. The sharpest reconstruction of the nine runs: MSE 0.00221, SSIM 0.831.*

<img src="report_figures/exp9_error.png" alt="Experiment 9 error maps" width="980" style="max-width:100%; height:auto;">

*Same patient0010 as the error maps of experiments 5 to 8. The orange profile follows the peaks, including both walls on the 4-chamber views. The ×5 error stays mostly on the sector border and on the wall.*

<img src="report_figures/exp9_latent.png" alt="Experiment 9 latent space" width="980" style="max-width:100%; height:auto;">

*z0 against cavity area: Spearman 0.95, Pearson 0.92. The four view / instant pairs follow the same line.*

<img src="report_figures/exp9_interp.png" alt="ED to ES interpolation, experiment 9" width="980" style="max-width:100%; height:auto;">

*patient0010, 4-chamber view, test set. Linear path from μ(ED) to μ(ES). The cavity closes without breaking the frame.*

<img src="report_figures/exp9_interp.gif" alt="ED to ES animation, experiment 9" width="300" style="max-width:100%; height:auto;">

*The same interpolation, as the animation saved by the run.*

<img src="report_figures/exp9_sweep.png" alt="z0 sweep, experiment 9" width="980" style="max-width:100%; height:auto;">

*Only channel 0 is moved, from the low end of its range to the high end, starting from a test frame with a median cavity area. This is the axis whose Spearman with the cavity is 0.953.*

<img src="report_figures/exp9_sweep.gif" alt="z0 sweep animation, experiment 9" width="300" style="max-width:100%; height:auto;">

*The same trajectory, as the animation saved by the run.*

<a id="comparison"></a>

## 8. Comparison

Lower is better for MSE, MAE and duration. Higher is better for PSNR, SSIM and Spearman. The best value of each row is in bold. A dash means the quantity was not measured. For \(z_1\) and \(z_2\), the bold value is the best among the runs that supervise that axis. Experiment 9 does not supervise them (0.669 and −0.315, section 7.9).

| | Exp 1 | Exp 2 | Exp 3 | Exp 4 | Exp 5 | Exp 6 | Exp 7 | Exp 8 | Exp 9 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Device | CPU | CPU | CPU | RTX 4060 Ti | RTX 4060 Ti | RTX 4060 Ti | RTX 4060 Ti | RTX 4060 Ti | RTX 4060 Ti |
| Latent | 4×8×8 | 8×8×8 | 8×8×8 | 8×8×8 | 8×8×8 | 16×8×8 | 12×8×8 | 8×8×8 | 8×8×8 |
| Parameters | 0.47 M | 1.89 M | 0.48 M | 3.26 M | 8.23 M | 49.1 M | 8.25 M | 8.23 M | 8.23 M |
| Epochs (best) | 25 (25) | 20 (19) | 20 (20) | 80 (76) | 100 (48) | 150 (150) | 120 (120) | 200 (193) | 110 (109) |
| Duration | 26.8 min | 30.5 min | 12.6 min | 3.8 min | 6.3 min | 39.3 min | 7.8 min | 11.6 min | 6.9 min |
| MSE | 0.01024 ± 0.01093 | 0.00765 ± 0.00538 | 0.01024 ± 0.01042 | 0.00343 ± 0.00261 | 0.00268 ± 0.00138 | 0.00609 ± 0.00281 | 0.00716 ± 0.01045 | 0.00439 ± 0.00297 | **0.00221 ± 0.00075** |
| Heart MSE | 0.01511 ± 0.01215 | 0.01073 ± 0.00486 | 0.01298 ± 0.01028 | 0.00518 ± 0.00206 | 0.00446 ± 0.00159 | 0.00842 ± 0.00315 | 0.00942 ± 0.01082 | 0.00639 ± 0.00271 | **0.00375 ± 0.00123** |
| MAE | 0.057 ± 0.026 | 0.055 ± 0.016 | 0.061 ± 0.026 | 0.034 ± 0.010 | 0.028 ± 0.006 | 0.042 ± 0.009 | 0.044 ± 0.022 | 0.036 ± 0.010 | **0.025 ± 0.004** |
| PSNR (dB) | 20.97 ± 2.68 | 21.70 ± 1.92 | 20.84 ± 2.48 | 25.12 ± 1.70 | 26.04 ± 1.50 | 22.49 ± 1.63 | 22.47 ± 2.18 | 23.99 ± 1.63 | **26.76 ± 1.29** |
| Heart PSNR (dB) | — | — | — | — | 23.73 ± 1.33 | 21.00 ± 1.46 | 21.04 ± 2.01 | 22.21 ± 1.42 | **24.46 ± 1.27** |
| SSIM | 0.622 ± 0.070 | 0.445 ± 0.065 | 0.703 ± 0.052 | 0.787 ± 0.030 | 0.808 ± 0.022 | 0.682 ± 0.033 | 0.715 ± 0.046 | 0.765 ± 0.028 | **0.831 ± 0.018** |
| Spearman(z0, LV) | — | — | 0.023 | 0.943 | 0.933 | 0.156 | 0.918 | 0.930 | **0.953** |
| Spearman(z1, myocardium) | — | — | — | — | — | 0.018 | 0.849 | **0.880** | — |
| Spearman(z2, left atrium) | — | — | — | — | — | 0.220 | **0.702** | 0.642 | — |


<a id="decision"></a>

## 9. What we keep

We keep **experiment 9**, epoch 109, file `checkpoints/exp9_vae_best.pth`.

It is at once the best reconstruction (MSE, heart MSE, MAE, PSNR, heart PSNR, SSIM) and the strongest cavity axis of the nine runs (Spearman 0.953). Experiment 8 still has the best supervised myocardium axis (0.880), and experiment 7 the best atrium axis (0.702). Both are the wrong model if the goal is the image: 22.47 dB and 23.99 dB, against 26.76 dB.

We do not keep experiment 6. Doubling the channels, doubling the width, adding a second attention scale, a spectral loss and a moving average did not beat an 8 M parameter model whose optimizer could actually move.

We do not keep experiment 7. The latent diagonal is real (0.918, 0.849, 0.702). The reconstruction did not follow: test MSE 0.00716, validation 0.0746 against 0.0613 for experiment 5.

We do not keep experiment 8. Bilinear augmentation on 64×64 smoothed the targets, the learning rate was already low when the sharp frames came back, and the frozen encoder locked that code. Validation 0.0721. The image checks all fail.

The experiment 5 file stays, `checkpoints/exp5_vae_best.pth`, because every later image check is written against it. Experiment 9 passes those checks: MSE 0.00221 against 0.00268, heart MSE 0.00375 against 0.00446, SSIM 0.831 against 0.808, PSNR 26.76 dB against 26.04 dB, Spearman(z0) 0.953 against 0.933. Validation at the checkpoint: 0.0553 against 0.0613.

What experiment 9 does not do:

- It does not supervise the myocardium or the left atrium. The measured Spearman values are 0.669 and −0.315.
- It does not segment. The mask is an input of the loss and of the scores, not an output of the network.
- The test set was not used to choose β, γ, or the epoch. It was used, across the project, to decide which experiment to write up. One look at the test set per experiment.

We stop on experiment 9.


<a id="bibliography"></a>

## 10. Bibliography

- S. Leclerc, E. Smistad, J. Pedrosa, A. Ostvik, et al. *Deep Learning for Segmentation using an Open Large-Scale Dataset in 2D Echocardiography.* IEEE Transactions on Medical Imaging, 38(9), 2198–2210, 2019. The CAMUS dataset.
- O. Bernard. *Assignment 2 — reconstructing CAMUS echocardiography with a VAE.* IAV, INSA Lyon. The course skeleton this project starts from.
- D. P. Kingma and M. Welling. *Auto-Encoding Variational Bayes.* ICLR, 2014. The ELBO, \(z = \mu + \varepsilon\sigma\), and the KL term used in every experiment.
- R. Rombach, A. Blattmann, D. Lorenz, P. Esser, and B. Ommer. *High-Resolution Image Synthesis with Latent Diffusion Models.* CVPR, 2022. The `AutoencoderKL` design MONAI implements.
- W. H. L. Pinaya et al. *Brain Imaging Generation with Latent Diffusion Models.* MICCAI Workshop, 2022. The same backbone on medical images.
- A. Pati and A. Lerch. *Attribute-based Regularization of Latent Spaces for Variational Auto-Encoders.* Neural Computing and Applications, 2021. The pairwise ranking loss behind γ from experiment 3 on.
- Z. Wang, A. C. Bovik, H. R. Sheikh, and E. P. Simoncelli. *Image Quality Assessment: From Error Visibility to Structural Similarity.* IEEE Transactions on Image Processing, 13(4), 2004. The SSIM.
- I. Loshchilov and F. Hutter. *Decoupled Weight Decay Regularization.* ICLR, 2019. AdamW, used from experiment 4 on.
- MONAI, *AutoencoderKL* and *SSIMLoss*. https://docs.monai.io. The architecture, and the SSIM (window 7, data range 1).
